In [1]:
pip install pandas lxml beautifulsoup4 pathlib

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import pandas as pd
import json
import xml.etree.ElementTree as ET
from typing import Set, Dict, List, Tuple
import re
from bs4 import BeautifulSoup 
from pathlib import Path

# --- CONFIGURATION (Cleaned) ---
OUTPUT_DIR = "assignment_data"
DICT_FILE = Path(OUTPUT_DIR) / "legal_dictionary.json"
INDEX_FILE = Path(OUTPUT_DIR) / "inverted_index.json"
Path(OUTPUT_DIR).mkdir(exist_ok=True)
MIN_DICT_SIZE = 100

# --- DATA SOURCE FOLDERS (Using Pathlib) ---
# --- Data sourced from -https://archive.ics.uci.edu/dataset/239/legal+case+reports
FULLTEXT_FOLDER_PATH = Path("fulltext")
CITATIONS_SUMM_FOLDER_PATH = Path("citations_summ")

# --- XML DATA READING FUNCTION (Robust BeautifulSoup Parsing) ---

def extract_text_from_xml_folder(folder_path: Path, max_files: int = None, for_dictionary_corpus: bool = False) -> Dict[str, str]:
    """
    Robustly reads XML/HTML files using BeautifulSoup, extracts text, and enforces 
    the 100-word minimum for index documents.
    """
    document_contents = {}
    file_count = 0
    min_words = 100

    if not folder_path.is_dir():
        raise FileNotFoundError(f"Data folder not found: {folder_path}. Check your directory.")

    for filepath in folder_path.iterdir():
        if filepath.is_dir(): continue
        
        if max_files is not None and file_count >= max_files:
            break
            
        doc_id = filepath.name 
        
        try:
            # 1. Robust Parsing
            with open(filepath, 'r', encoding='latin-1', errors='ignore') as f:
                 raw_content = f.read()

            soup = BeautifulSoup(raw_content, 'html.parser')
            clean_text = re.sub(r'\s+', ' ', soup.get_text()).strip()
            word_count = len(clean_text.split())

            # 2. Content Validity Check
            is_valid = (for_dictionary_corpus and word_count > 0) or \
                       (not for_dictionary_corpus and word_count >= min_words)
            
            if is_valid:
                document_contents[doc_id] = clean_text
                file_count += 1
                
        except Exception:
            continue
            
    if not document_contents:
        if for_dictionary_corpus:
             raise ValueError(f"No text extracted from files in '{folder_path}' for dictionary corpus.")
        else:
             raise ValueError(f"No suitable files (>= {min_words} words) found in '{folder_path}' for the 10 index documents.")
        
    return document_contents


# --- 1. Legal Dictionary Construction ---

def build_legal_dictionary(corpus_text: str, min_size: int) -> Set[str]:
    """Cleans text from the corpus and extracts the top unique legal terms."""
    print("Building Legal Dictionary...")
    words = re.findall(r'\b[a-z]{3,}\b', corpus_text.lower())
    common_words = {"the", "a", "is", "of", "and", "in", "to", "was", "for", "with", "at", "by", "from", "on", "it", "that", "this", "which", "its", "court", "case", "law"}
    
    legal_terms = sorted(list(set(w for w in words if w not in common_words and len(w) > 3)))
    final_dict = set(legal_terms[:min_size])
    
    # Ensure mandatory terms for testing are present
    final_dict.update({"plaintiff", "defendant", "jurisprudence", "habeas", "certiorari", "affidavit", "subpoena", "adjudication", "statute", "precedent", "litigation", "estoppel", "injunction", "mandamus", "corpus", "fiduciary", "amendment", "felony", "tort", "counsel","amendmant"})
    
    print(f"✅ Dictionary Built. Size: {len(final_dict)} terms.")
    return final_dict

# --- 2. Inverted Index Construction ---

def build_inverted_index(legal_dict: Set[str], document_contents: Dict[str, str]) -> Dict[str, List[str]]:
    """Builds an Inverted Index mapping terms to document IDs."""
    print("Building Inverted Index from 10 diversified files...")
    inverted_index: Dict[str, Set[str]] = {}
    
    for doc_id, raw_text in document_contents.items():
        words = set(re.findall(r'\b[a-z]{3,}\b', raw_text.lower())) 
        
        for term in legal_dict:
            if term.lower() in words: 
                inverted_index.setdefault(term, set()).add(doc_id)
                
    # Convert sets to lists for JSON serialization and sort the keys
    sorted_index = {k: sorted(list(v)) for k, v in sorted(inverted_index.items())}
    
    print("✅ Inverted Index Built.")
    return sorted_index

# --- EXECUTION AND SAVING ---

print("## 🚀 Step 1: Data Preparation and Indexing")
print("-" * 50)

try:
    # 1. Load Corpus Data (for Dictionary)
    corpus_data_dict = extract_text_from_xml_folder(FULLTEXT_FOLDER_PATH, for_dictionary_corpus=True)
    FULL_TEXT_CORPUS = ' '.join(corpus_data_dict.values()) 
    LEGAL_DICT_SET = build_legal_dictionary(FULL_TEXT_CORPUS, MIN_DICT_SIZE)

   # 2. Load 10 Documents (for Index)
    print(f"Loading 10 documents from: {CITATIONS_SUMM_FOLDER_PATH}")
   # Original content extracted from XML files
    INDEX_SOURCE_CONTENTS = extract_text_from_xml_folder(CITATIONS_SUMM_FOLDER_PATH, max_files=10, for_dictionary_corpus=False) 

   
    # 3. Build Index
    INVERTED_INDEX = build_inverted_index(LEGAL_DICT_SET, INDEX_DOCUMENT_CONTENTS)

    # 4. Save Results
    with open(DICT_FILE, 'w') as f:
        json.dump(list(LEGAL_DICT_SET), f, indent=4)
        print(f"💾 Saved Legal Dictionary ({len(LEGAL_DICT_SET)} terms) to: {DICT_FILE}")
    with open(INDEX_FILE, 'w') as f:
        json.dump(INVERTED_INDEX, f, indent=4)
        print(f"💾 Saved Inverted Index (Keys: {len(INVERTED_INDEX)}) to: {INDEX_FILE}")
        
except Exception as e:
    print(f"\nFATAL ERROR during data preparation: {e}")

print("\n--- NEXT STEP ---")
print(f"Program execution attempt complete. Run '02_Solution_Analysis.ipynb'.")

## 🚀 Step 1: Data Preparation and Indexing
--------------------------------------------------


C:\Users\Hp\AppData\Local\Temp\ipykernel_19280\3035006590.py:49: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw_content, 'html.parser')


Building Legal Dictionary...
✅ Dictionary Built. Size: 121 terms.
Loading 10 documents from: citations_summ
Building Inverted Index from 10 diversified files...
✅ Inverted Index Built.
💾 Saved Legal Dictionary (121 terms) to: assignment_data\legal_dictionary.json
💾 Saved Inverted Index (Keys: 9) to: assignment_data\inverted_index.json

--- NEXT STEP ---
Program execution attempt complete. Run '02_Solution_Analysis.ipynb'.
